# 02 — Exploratory Data Analysis

Produces reproducible figures and Power BI-ready summary tables. Every insight must cite an artifact generated here.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore")
current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (current, *current.parents) if (p / "config" / "config.yaml").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate project root containing config/config.yaml")
with (PROJECT_ROOT / "config" / "config.yaml").open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

def project_path(key):
    return PROJECT_ROOT / CONFIG["paths"][key]

OUTPUT = project_path("output")
PROCESSED = project_path("processed")
for folder in [OUTPUT / "reports", OUTPUT / "figures", OUTPUT / "models", OUTPUT / "labeling", OUTPUT / "logs", OUTPUT / "powerbi", PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT / ".matplotlib"))
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

shops = pd.read_csv(PROCESSED / "shops_clean.csv")
products = pd.read_csv(PROCESSED / "products_clean.csv")
reviews = pd.read_csv(PROCESSED / "reviews_clean.csv")
sns.set_theme(style="whitegrid")

def save(fig, name):
    fig.tight_layout(); fig.savefig(OUTPUT / "figures" / name, dpi=160, bbox_inches="tight"); plt.show(); plt.close(fig)
def histogram(frame, column, name):
    values = frame[column].dropna()
    if values.empty: print(f"SKIP {column}: no observed values"); return
    fig, ax = plt.subplots(); sns.histplot(values, bins=min(25,max(5,len(values))), ax=ax); ax.set_title(column); save(fig,name)

for frame, column, name in [(shops,"Follower_Count_k","followers.png"),(shops,"Rating_Average","shop_rating.png"),(shops,"Chat_Response_Rate_pct","response_rate.png"),(products,"Review_stars","product_rating.png"),(reviews,"rating","review_rating.png"),(reviews,"Comment_Length","comment_length.png")]: histogram(frame,column,name)
if not reviews["has_image"].dropna().empty:
    fig, ax = plt.subplots(); reviews["has_image"].value_counts().sort_index().plot.pie(autopct="%1.1f%%", ax=ax); ax.set_ylabel(""); ax.set_title("Review image rate"); save(fig,"has_image.png")

In [ ]:
def scatter(x, y, name, title):
    if x not in shops or y not in shops or shops[[x,y]].dropna().empty: return
    fig, ax = plt.subplots(); sns.regplot(data=shops, x=x, y=y, ci=None, ax=ax); ax.set_title(title); save(fig,name)
scatter("Rating_Average","Sales","rating_sales.png","Rating vs observed sales")
scatter("Chat_Response_Rate_pct","Conversion_Rate_pct","response_conversion.png","Response rate vs conversion")
scatter("Follower_Count_k","Rating_Average","follower_rating.png","Followers vs rating")

for label in ["positive","negative"]:
    text = " ".join(reviews.loc[reviews["Sentiment_Label"] == label, "review_clean"].dropna().astype(str))
    if text.strip():
        cloud = WordCloud(width=1200,height=600,background_color="white",collocations=False).generate(text)
        fig, ax = plt.subplots(figsize=(12,6)); ax.imshow(cloud); ax.axis("off"); ax.set_title(f"{label.title()} review keywords"); save(fig,f"wordcloud_{label}.png")

summary = pd.DataFrame({"Metric":["Total Shops","Total Products","Total Reviews","Average Review Rating","Image Rate","Observed Units Sold"],"Value":[len(shops),len(products),len(reviews),reviews["rating"].mean(),reviews["has_image"].mean(),products.get("units_sold",pd.Series(dtype=float)).sum()]})
shop_eda = shops.copy()
shop_eda["Product_Count"] = shop_eda["Shop_ID"].map(products.groupby("Shop_ID").size()).fillna(0)
shop_eda["Review_Count"] = shop_eda["Shop_ID"].map(reviews.groupby("Shop_ID").size()).fillna(0)
shop_eda["Review_Image_Rate"] = shop_eda["Shop_ID"].map(reviews.groupby("Shop_ID")["has_image"].mean())
summary.to_csv(OUTPUT / "powerbi" / "kpi_summary.csv", index=False)
shop_eda.to_csv(OUTPUT / "powerbi" / "shop_eda_summary.csv", index=False)
reviews.groupby(["Shop_ID","Sentiment_Label"], observed=True).size().rename("Review_Count").reset_index().to_csv(OUTPUT / "powerbi" / "sentiment_summary.csv", index=False)
display(summary); display(shop_eda)